In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Task 1: Write your code here:

p1 = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(p1)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
df["Delivery_Time"].hist()


In [ ]:
# Task 1: Write your code here:
df = df.drop("Order_ID",axis=1)



In [ ]:
# Task 2: Write your code here:
# Missing values
print("Missing values:")
print(df.isnull().sum())

In [ ]:

df["Delivery_Time"]=df["Delivery_Time"].fillna(df['Delivery_Time'].mean())

df["Weather"]=df["Weather"].fillna(df['Weather'].mode())

In [ ]:
# Define stat columns
stat_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs']

df["Weather"]=df["Weather"].fillna(df["Weather"].mode()[0])
df["Traffic_Level"]=df["Traffic_Level"].fillna(df["Traffic_Level"].mode()[0])
df["Time_of_Day"]=df["Time_of_Day"].fillna(df["Time_of_Day"].mode()[0])
df["Courier_Experience_yrs"]=df["Courier_Experience_yrs"].fillna(df["Courier_Experience_yrs"].mode()[0])


In [ ]:
df

In [ ]:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Task 3: Write your code here:
#  Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
# 3. Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:
# Task 5: Write your code here:
features = df.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 6: Write your code here:
df["Delivery_Time"].hist()

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:

from sklearn.ensemble import RandomForestRegressor



In [ ]:
models = {

  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),


}

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score, mean_absolute_error as mae


In [ ]:
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
# Storage for linear regression results for each fold
# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mae': []}


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae1 = mae(y_test, y_pred)


    # Store results
    all_results[model_name]["mae"].append(mae1)


In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  mae:  {np.mean(all_results[model_name]['mae']):.4f}")



In [ ]:
# Calculate the baseline predictions (mean of the target)
baseline_pred = np.full_like(y, y.median())

# =================we using median becuse it is mae :)

# Evaluate the baseline

baseline_mae= mae(y, baseline_pred)


print(f"Baseline MSE (using mean target): {baseline_mae:.4f}")


In [ ]:
# Task 1: Write your code here:
# Gather importances from the models (from the last fold)
importances = {}


importances['Random Forest Regressor'] = models['Random Forest Regressor'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(30, 50))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
import seaborn as sns
sns.heatmap(df.drop("Delivery_Time",axis=1).corr(), annot=True)

In [ ]:
# Task Bonus: Write your code here:
